# Practical Semantic Compliance Prototype (OWL/SHACL-oriented)

この notebook は、Knowledge Engineer / AI Researcher (Ontologies & Semantic Reasoning) 系の JD を意識して、
**規格条文の形式化 → SHACL shape 生成 → 設計グラフ検証 → clause-level compliance verdict**
までを一通り体験できるようにした、やや実務寄りの最小プロトタイプです。

## この notebook でやること

1. **規格条文グラフ**を Turtle で定義する
2. **設計アーティファクト側グラフ**を Turtle で定義する
3. 条文グラフから **SHACL shape を自動生成**する
4. pySHACL で validation を実行する
5. validation report をもとに **clause-level verdict table** を作る
6. どの clause が、どの artifact に対して、なぜ violated / compliant なのかを確認する

## 題材

安全系設備をかなり簡略化した例として、次の規格文を扱います。

- **Clause 5.2.1**: A fire door shall have exactly one identifier.
- **Clause 5.2.2**: A fire door shall be connected to at least one alarm zone.
- **Clause 5.2.3**: A fire door should have at least one inspection record.

ここでは、`shall` は violation の対象、`should` は warning 的な扱いを想定します。


## 0. 必要ライブラリ

- `rdflib`
- `pandas`
- `pyshacl`

`pyshacl` が未導入なら、次のセルを実行してください。


In [ ]:
# 必要なら実行
%pip install pyshacl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 4.8 MB/s eta 0:00:00


In [ ]:
from rdflib import Graph, Namespace, RDF, URIRef
from rdflib.namespace import SH, XSD
import pandas as pd

try:
    from pyshacl import validate
except ImportError as e:
    raise ImportError(
        "pyshacl が見つかりません。必要なら先頭の `%pip install pyshacl` セルを実行してください。"
    ) from e

EX = Namespace('http://example.org/')
STD = Namespace('http://example.org/standard/')
ART = Namespace('http://example.org/artifact/')


## 1. 規格条文グラフを定義する

ここでは、**条文それ自体** を graph 化します。  
ポイントは、単に FireDoor などの ontology を作るだけでなく、
**Clause / modality / appliesTo / requiresProperty / cardinality / targetClass** を明示することです。

この層があると、

- どの shape がどの clause 由来か
- どの verdict がどの clause に紐づくか
- 規格文グラフと設計グラフをどう接続するか

が見やすくなります。


In [ ]:
standard_ttl = '''
@prefix ex: <http://example.org/> .
@prefix std: <http://example.org/standard/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

# --- domain classes ---
ex:FireDoor a rdfs:Class .
ex:AlarmZone a rdfs:Class .
ex:InspectionRecord a rdfs:Class .

# --- domain properties ---
ex:hasIdentifier a rdf:Property .
ex:connectedTo a rdf:Property .
ex:hasInspectionRecord a rdf:Property .

# --- standard meta-model ---
std:Clause a rdfs:Class .
std:ObligationLevel a rdfs:Class .
std:Shall a std:ObligationLevel .
std:Should a std:ObligationLevel .

std:appliesTo a rdf:Property .
std:requiresProperty a rdf:Property .
std:requiresTargetClass a rdf:Property .
std:minCount a rdf:Property .
std:maxCount a rdf:Property .
std:hasDatatype a rdf:Property .
std:hasModality a rdf:Property .
std:clauseText a rdf:Property .
std:clauseCode a rdf:Property .
std:derivedFromClause a rdf:Property .

# --- Clause 5.2.1 ---
std:Clause_5_2_1 a std:Clause ;
    std:clauseCode "5.2.1" ;
    std:clauseText "A fire door shall have exactly one identifier." ;
    std:hasModality std:Shall ;
    std:appliesTo ex:FireDoor ;
    std:requiresProperty ex:hasIdentifier ;
    std:minCount 1 ;
    std:maxCount 1 ;
    std:hasDatatype xsd:string .

# --- Clause 5.2.2 ---
std:Clause_5_2_2 a std:Clause ;
    std:clauseCode "5.2.2" ;
    std:clauseText "A fire door shall be connected to at least one alarm zone." ;
    std:hasModality std:Shall ;
    std:appliesTo ex:FireDoor ;
    std:requiresProperty ex:connectedTo ;
    std:minCount 1 ;
    std:requiresTargetClass ex:AlarmZone .

# --- Clause 5.2.3 ---
std:Clause_5_2_3 a std:Clause ;
    std:clauseCode "5.2.3" ;
    std:clauseText "A fire door should have at least one inspection record." ;
    std:hasModality std:Should ;
    std:appliesTo ex:FireDoor ;
    std:requiresProperty ex:hasInspectionRecord ;
    std:minCount 1 ;
    std:requiresTargetClass ex:InspectionRecord .
'''

g_standard = Graph()
g_standard.parse(data=standard_ttl, format='turtle')
print('standard triples:', len(g_standard))


standard triples: 45


## 2. 設計アーティファクト側グラフを定義する

ここでは、設計対象側の graph をかなり簡略化して定義します。

- `FD001` は identifier も alarm zone 接続もあり、inspection record もある
- `FD002` は identifier が2つあり、inspection record が無い
- `FD003` は identifier はあるが、alarm zone 接続が無い


In [ ]:
artifact_ttl = '''
@prefix ex: <http://example.org/> .
@prefix art: <http://example.org/artifact/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

art:ZoneA rdf:type ex:AlarmZone .
art:Inspect_001 rdf:type ex:InspectionRecord .

art:FD001 rdf:type ex:FireDoor ;
    ex:hasIdentifier "FD001" ;
    ex:connectedTo art:ZoneA ;
    ex:hasInspectionRecord art:Inspect_001 .

art:FD002 rdf:type ex:FireDoor ;
    ex:hasIdentifier "FD002-A" ;
    ex:hasIdentifier "FD002-B" ;
    ex:connectedTo art:ZoneA .

art:FD003 rdf:type ex:FireDoor ;
    ex:hasIdentifier "FD003" .
'''

g_artifact = Graph()
g_artifact.parse(data=artifact_ttl, format='turtle')
print('artifact triples:', len(g_artifact))


artifact triples: 12


## 3. triples を table として確認する

In [ ]:
def graph_to_df(graph: Graph) -> pd.DataFrame:
    rows = [(str(s), str(p), str(o)) for s, p, o in graph]
    return pd.DataFrame(rows, columns=['subject', 'predicate', 'object'])

standard_df = graph_to_df(g_standard).sort_values(['subject', 'predicate', 'object']).reset_index(drop=True)
artifact_df = graph_to_df(g_artifact).sort_values(['subject', 'predicate', 'object']).reset_index(drop=True)

print('--- standard graph ---')
display(standard_df.head(50))
print('--- artifact graph ---')
display(artifact_df.head(50))


--- standard graph ---


,subject,predicate,object
0,http://example.org/AlarmZone,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
1,http://example.org/FireDoor,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
2,http://example.org/InspectionRecord,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
3,http://example.org/connectedTo,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...
4,http://example.org/hasIdentifier,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...
5,http://example.org/hasInspectionRecord,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...
6,http://example.org/standard/Clause,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
7,http://example.org/standard/Clause_5_2_1,http://example.org/standard/appliesTo,http://example.org/FireDoor
8,http://example.org/standard/Clause_5_2_1,http://example.org/standard/clauseCode,5.2.1
9,http://example.org/standard/Clause_5_2_1,http://example.org/standard/clauseText,A fire door shall have exactly one identifier.


--- artifact graph ---


,subject,predicate,object
0,http://example.org/artifact/FD001,http://example.org/connectedTo,http://example.org/artifact/ZoneA
1,http://example.org/artifact/FD001,http://example.org/hasIdentifier,FD001
2,http://example.org/artifact/FD001,http://example.org/hasInspectionRecord,http://example.org/artifact/Inspect_001
3,http://example.org/artifact/FD001,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/FireDoor
4,http://example.org/artifact/FD002,http://example.org/connectedTo,http://example.org/artifact/ZoneA
5,http://example.org/artifact/FD002,http://example.org/hasIdentifier,FD002-A
6,http://example.org/artifact/FD002,http://example.org/hasIdentifier,FD002-B
7,http://example.org/artifact/FD002,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/FireDoor
8,http://example.org/artifact/FD003,http://example.org/hasIdentifier,FD003
9,http://example.org/artifact/FD003,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/FireDoor


## 4. 条文グラフから SHACL shape を自動生成する

今回は hand-written の SHACL を書かず、**Clause graph を読んで shape graph を programmatically に作る** ようにします。

今回は単純化して、各 clause を 1 つの `NodeShape` とし、
`appliesTo`, `requiresProperty`, `minCount`, `maxCount`, `hasDatatype`, `requiresTargetClass`
から property constraint を組み立てます。


In [ ]:
# g_std: 規格条文のグラフ
# RDF graph を受け取って、別の RDF graph を返す
def build_shapes_from_standard(g_std: Graph) -> Graph:
    # SHACL を入れていくための空 graph を作成
    g_shapes = Graph()
    g_shapes.bind('sh', SH)
    g_shapes.bind('ex', EX)
    g_shapes.bind('std', STD)
    g_shapes.bind('xsd', XSD)

    import rdflib

    # 規格 graph の中から clause を抽出
    # rdf:type std:Clause を持つ subject を全部集める
    # つまり g_std の中にある
    # std:Clause_5_2_1
    # std:Clause_5_2_2
    # std:Clause_5_2_3
    # みたいな条文ノードを全部取得している
    clauses = list(g_std.subjects(RDF.type, STD.Clause))

    # 各 clause ごとに shape を1つ作る
    # ここでは各条文に対して、
    # Clause_5_2_1_Shape
    # Clause_5_2_2_Shape
    # みたいな shape URI を作成
    # そしてその shape は
    # shape rdf:type sh:NodeShape
    for clause in clauses:
        shape = URIRef(str(clause) + '_Shape')
        g_shapes.add((shape, RDF.type, SH.NodeShape))

        # 条文から必要な情報を読む
        # ex.
        # appliesTo = FireDoor
        # requiresProperty = hasIdentifier
        # minCount = 1
        # maxCount = 1
        # datatype = xsd:string
        # modality = Shall
        # clauseCode = "5.2.1"
        applies_to = g_std.value(clause, STD.appliesTo)
        req_prop = g_std.value(clause, STD.requiresProperty)
        min_count = g_std.value(clause, STD.minCount)
        max_count = g_std.value(clause, STD.maxCount)
        datatype = g_std.value(clause, STD.hasDatatype)
        target_class = g_std.value(clause, STD.requiresTargetClass)
        modality = g_std.value(clause, STD.hasModality)
        clause_code = g_std.value(clause, STD.clauseCode)

        # この shape がどの class に適用されるかを設定
        if applies_to is not None:
            g_shapes.add((shape, SH.targetClass, applies_to))

        # blank node を作成
        # SHACL の property constraint は、その場限りの無名ノードで表すことが多い
        pshape = rdflib.BNode()
        g_shapes.add((shape, SH.property, pshape))
        # SHACL では、NodeShape の中に
        # どの property を検査するか
        # その property にどんな制約があるか
        # を書く必要があります。
        # property 制約の箱として pshape という blank node を作っている
        g_shapes.add((pshape, SH.path, req_prop))

        # 件数・型・対象クラスなどの制約を追加
        if min_count is not None:
            g_shapes.add((pshape, SH.minCount, min_count))
        if max_count is not None:
            g_shapes.add((pshape, SH.maxCount, max_count))
        if datatype is not None:
            g_shapes.add((pshape, SH.datatype, datatype))
        if target_class is not None:
            g_shapes.add((pshape, SH['class'], target_class))

        if clause_code is not None:
            g_shapes.add((shape, STD.clauseCode, clause_code))
        if modality is not None:
            g_shapes.add((shape, STD.hasModality, modality))
        g_shapes.add((shape, STD.derivedFromClause, clause))

        # shall と should で severity を分ける
        # 今回は簡略化して、
        # should → Warning
        # それ以外（実質 shall）→ Violation
        # にしています
        if modality == STD.Should:
            g_shapes.add((shape, SH.severity, SH.Warning))
        else:
            g_shapes.add((shape, SH.severity, SH.Violation))

    return g_shapes

g_shapes = build_shapes_from_standard(g_standard)
print('generated shape triples:', len(g_shapes))


generated shape triples: 31


## 5. 生成された SHACL graph を Turtle で確認する

In [ ]:
print(g_shapes.serialize(format='turtle'))


@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix std: <http://example.org/standard/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

std:Clause_5_2_1_Shape a sh:NodeShape ;
    std:clauseCode "5.2.1" ;
    std:derivedFromClause std:Clause_5_2_1 ;
    std:hasModality std:Shall ;
    sh:property [ sh:datatype xsd:string ;
            sh:maxCount 1 ;
            sh:minCount 1 ;
            sh:path ex:hasIdentifier ] ;
    sh:severity sh:Violation ;
    sh:targetClass ex:FireDoor .

std:Clause_5_2_2_Shape a sh:NodeShape ;
    std:clauseCode "5.2.2" ;
    std:derivedFromClause std:Clause_5_2_2 ;
    std:hasModality std:Shall ;
    sh:property [ sh:class ex:AlarmZone ;
            sh:minCount 1 ;
            sh:path ex:connectedTo ] ;
    sh:severity sh:Violation ;
    sh:targetClass ex:FireDoor .

std:Clause_5_2_3_Shape a sh:NodeShape ;
    std:clauseCode "5.2.3" ;
    std:derivedFromClause std:Clause_5_2_3 ;
    std:hasModality std:Should ;
    

## 6. SHACL validation を実行する

In [ ]:
conforms, results_graph, results_text = validate(
    data_graph=g_artifact,
    shacl_graph=g_shapes,
    inference='rdfs',
    abort_on_first=False,
    allow_infos=True,
    allow_warnings=True,
)

print('Overall conforms:', conforms)
print(results_text)


Overall conforms: False
Validation Report
Conforms: False
Results (4):
Constraint Violation in MaxCountConstraintComponent (http://www.w3.org/ns/shacl#MaxCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:string ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ex:hasIdentifier ]
	Focus Node: art:FD002
	Result Path: ex:hasIdentifier
	Message: More than 1 values on art:FD002->ex:hasIdentifier
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class ex:AlarmZone ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ex:connectedTo ]
	Focus Node: art:FD003
	Result Path: ex:connectedTo
	Message: Less than 1 values on art:FD003->ex:connectedTo
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:c

## 7. Validation result graph を table にする

In [ ]:
results_df = graph_to_df(results_graph).sort_values(['subject', 'predicate', 'object']).reset_index(drop=True)
display(results_df.head(100))


,subject,predicate,object
0,N3dab3b1eca294487956483ee61d4c2e5,http://www.w3.org/ns/shacl#datatype,http://www.w3.org/2001/XMLSchema#string
1,N3dab3b1eca294487956483ee61d4c2e5,http://www.w3.org/ns/shacl#maxCount,1
2,N3dab3b1eca294487956483ee61d4c2e5,http://www.w3.org/ns/shacl#minCount,1
3,N3dab3b1eca294487956483ee61d4c2e5,http://www.w3.org/ns/shacl#path,http://example.org/hasIdentifier
4,N5308ec3ebad44817948cc476dec4deab,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/ns/shacl#ValidationReport
5,N5308ec3ebad44817948cc476dec4deab,http://www.w3.org/ns/shacl#conforms,false
6,N5308ec3ebad44817948cc476dec4deab,http://www.w3.org/ns/shacl#result,N7cd3abe6338a4a7da334e8a00a08b4ba
7,N5308ec3ebad44817948cc476dec4deab,http://www.w3.org/ns/shacl#result,Ndb21aaef29244e2c8a55d6e2d27f6de6
8,N5308ec3ebad44817948cc476dec4deab,http://www.w3.org/ns/shacl#result,Ne5872d385463414d85d8f4b1f872a7f5
9,N5308ec3ebad44817948cc476dec4deab,http://www.w3.org/ns/shacl#result,Ne79574470e03489e98bf3b4e09fcfdb3


## 8. Clause-level verdict table を構築する

ここでは、SHACL report から

- `focus node` (どの artifact か)
- `source shape` (どの clause 由来の shape か)
- `severity`
- `message`

を抜き出し、さらに `shape -> clause` の対応を使って、
**artifact × clause 単位の verdict table** を作ります。


In [ ]:
def literal_to_str(x):
    return None if x is None else str(x)

shape_meta_rows = []
for shape in g_shapes.subjects(RDF.type, SH.NodeShape):
    clause_uri = g_shapes.value(shape, STD.derivedFromClause)
    clause_code = None
    clause_text = None
    modality = None
    if clause_uri is not None:
        clause_code = g_standard.value(clause_uri, STD.clauseCode)
        clause_text = g_standard.value(clause_uri, STD.clauseText)
        modality = g_standard.value(clause_uri, STD.hasModality)
    shape_meta_rows.append({
        'shape': str(shape),
        'clause_uri': literal_to_str(clause_uri),
        'clause_code': literal_to_str(clause_code),
        'clause_text': literal_to_str(clause_text),
        'modality': literal_to_str(modality),
    })

shape_meta_df = pd.DataFrame(shape_meta_rows)
display(shape_meta_df)

report_rows = []
for vr in results_graph.subjects(RDF.type, SH.ValidationResult):
    focus_node = results_graph.value(vr, SH.focusNode)
    source_shape = results_graph.value(vr, SH.sourceShape)
    severity = results_graph.value(vr, SH.resultSeverity)
    message = results_graph.value(vr, SH.resultMessage)
    result_path = results_graph.value(vr, SH.resultPath)
    value = results_graph.value(vr, SH.value)
    report_rows.append({
        'validation_result': str(vr),
        'focus_node': literal_to_str(focus_node),
        'source_shape': literal_to_str(source_shape),
        'severity': literal_to_str(severity),
        'result_path': literal_to_str(result_path),
        'value': literal_to_str(value),
        'message': literal_to_str(message),
    })

report_df = pd.DataFrame(report_rows)
display(report_df)

if len(report_df) > 0:
    clause_report_df = report_df.merge(shape_meta_df, left_on='source_shape', right_on='shape', how='left')
else:
    clause_report_df = report_df.copy()

display(clause_report_df)


,shape,clause_uri,clause_code,clause_text,modality
0,http://example.org/standard/Clause_5_2_1_Shape,http://example.org/standard/Clause_5_2_1,5.2.1,A fire door shall have exactly one identifier.,http://example.org/standard/Shall
1,http://example.org/standard/Clause_5_2_2_Shape,http://example.org/standard/Clause_5_2_2,5.2.2,A fire door shall be connected to at least one...,http://example.org/standard/Shall
2,http://example.org/standard/Clause_5_2_3_Shape,http://example.org/standard/Clause_5_2_3,5.2.3,A fire door should have at least one inspectio...,http://example.org/standard/Should


,validation_result,focus_node,source_shape,severity,result_path,value,message
0,Ne5872d385463414d85d8f4b1f872a7f5,http://example.org/artifact/FD002,N3dab3b1eca294487956483ee61d4c2e5,http://www.w3.org/ns/shacl#Violation,http://example.org/hasIdentifier,None,More than 1 values on art:FD002->ex:hasIdentifier
1,Ne79574470e03489e98bf3b4e09fcfdb3,http://example.org/artifact/FD003,N794d683b03ed4625ae97c29f77fac22e,http://www.w3.org/ns/shacl#Violation,http://example.org/connectedTo,None,Less than 1 values on art:FD003->ex:connectedTo
2,Ndb21aaef29244e2c8a55d6e2d27f6de6,http://example.org/artifact/FD003,Ncb03f8b197984df4b9b0d025200ca54e,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,None,Less than 1 values on art:FD003->ex:hasInspect...
3,N7cd3abe6338a4a7da334e8a00a08b4ba,http://example.org/artifact/FD002,Ncb03f8b197984df4b9b0d025200ca54e,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,None,Less than 1 values on art:FD002->ex:hasInspect...


,validation_result,focus_node,source_shape,severity,result_path,value,message,shape,clause_uri,clause_code,clause_text,modality
0,Ne5872d385463414d85d8f4b1f872a7f5,http://example.org/artifact/FD002,N3dab3b1eca294487956483ee61d4c2e5,http://www.w3.org/ns/shacl#Violation,http://example.org/hasIdentifier,None,More than 1 values on art:FD002->ex:hasIdentifier,NaN,NaN,NaN,NaN,NaN
1,Ne79574470e03489e98bf3b4e09fcfdb3,http://example.org/artifact/FD003,N794d683b03ed4625ae97c29f77fac22e,http://www.w3.org/ns/shacl#Violation,http://example.org/connectedTo,None,Less than 1 values on art:FD003->ex:connectedTo,NaN,NaN,NaN,NaN,NaN
2,Ndb21aaef29244e2c8a55d6e2d27f6de6,http://example.org/artifact/FD003,Ncb03f8b197984df4b9b0d025200ca54e,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,None,Less than 1 values on art:FD003->ex:hasInspect...,NaN,NaN,NaN,NaN,NaN
3,N7cd3abe6338a4a7da334e8a00a08b4ba,http://example.org/artifact/FD002,Ncb03f8b197984df4b9b0d025200ca54e,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,None,Less than 1 values on art:FD002->ex:hasInspect...,NaN,NaN,NaN,NaN,NaN


## 9. Artifact × Clause の verdict を作る

validation report は violation / warning だけが出るので、
ここでは対象 artifact と対象 clause の全組み合わせを作り、

- report があるもの → violated / warning
- report が無いもの → compliant(違反していない)

として簡略な verdict table を作ります。


In [ ]:
firedoors = sorted({str(s) for s in g_artifact.subjects(RDF.type, EX.FireDoor)})

clause_rows = []
for clause in g_standard.subjects(RDF.type, STD.Clause):
    clause_rows.append({
        'clause_uri': str(clause),
        'clause_code': literal_to_str(g_standard.value(clause, STD.clauseCode)),
        'clause_text': literal_to_str(g_standard.value(clause, STD.clauseText)),
        'modality': literal_to_str(g_standard.value(clause, STD.hasModality)),
    })
clauses_df = pd.DataFrame(clause_rows).sort_values('clause_code').reset_index(drop=True)

all_pairs = pd.MultiIndex.from_product([firedoors, clauses_df['clause_code']], names=['focus_node', 'clause_code']).to_frame(index=False)

if len(clause_report_df) > 0:
    report_key_df = clause_report_df[['focus_node', 'clause_code', 'severity', 'message']].copy()
else:
    report_key_df = pd.DataFrame(columns=['focus_node', 'clause_code', 'severity', 'message'])

verdict_df = all_pairs.merge(report_key_df, on=['focus_node', 'clause_code'], how='left')
verdict_df = verdict_df.merge(clauses_df[['clause_code', 'clause_text', 'modality']], on='clause_code', how='left')

def assign_status(row):
    sev = row['severity']
    if pd.isna(sev):
        return 'compliant'
    if str(sev).endswith('Warning'):
        return 'warning'
    return 'violated'

verdict_df['status'] = verdict_df.apply(assign_status, axis=1)
verdict_df = verdict_df[['focus_node', 'clause_code', 'status', 'modality', 'clause_text', 'severity', 'message']]
verdict_df = verdict_df.sort_values(['focus_node', 'clause_code']).reset_index(drop=True)
display(verdict_df)


,focus_node,clause_code,status,modality,clause_text,severity,message
0,http://example.org/artifact/FD001,5.2.1,compliant,http://example.org/standard/Shall,A fire door shall have exactly one identifier.,NaN,NaN
1,http://example.org/artifact/FD001,5.2.2,compliant,http://example.org/standard/Shall,A fire door shall be connected to at least one...,NaN,NaN
2,http://example.org/artifact/FD001,5.2.3,compliant,http://example.org/standard/Should,A fire door should have at least one inspectio...,NaN,NaN
3,http://example.org/artifact/FD002,5.2.1,compliant,http://example.org/standard/Shall,A fire door shall have exactly one identifier.,NaN,NaN
4,http://example.org/artifact/FD002,5.2.2,compliant,http://example.org/standard/Shall,A fire door shall be connected to at least one...,NaN,NaN
5,http://example.org/artifact/FD002,5.2.3,compliant,http://example.org/standard/Should,A fire door should have at least one inspectio...,NaN,NaN
6,http://example.org/artifact/FD003,5.2.1,compliant,http://example.org/standard/Shall,A fire door shall have exactly one identifier.,NaN,NaN
7,http://example.org/artifact/FD003,5.2.2,compliant,http://example.org/standard/Shall,A fire door shall be connected to at least one...,NaN,NaN
8,http://example.org/artifact/FD003,5.2.3,compliant,http://example.org/standard/Should,A fire door should have at least one inspectio...,NaN,NaN


## 10. 集計してみる

In [ ]:
print('--- by artifact ---')
display(
    verdict_df.groupby(['focus_node', 'status']).size().unstack(fill_value=0).reset_index()
)

print('--- by clause ---')
display(
    verdict_df.groupby(['clause_code', 'status']).size().unstack(fill_value=0).reset_index()
)


--- by artifact ---


status,focus_node,compliant
0,http://example.org/artifact/FD001,3
1,http://example.org/artifact/FD002,3
2,http://example.org/artifact/FD003,3


--- by clause ---


status,clause_code,compliant
0,5.2.1,3
1,5.2.2,3
2,5.2.3,3


## 11. SPARQL で規格 graph を引く

規格 graph 側に対して、どの clause がどの property を要求しているかを SPARQL で確認します。

- SQL = 表に対する問い合わせ
- SPARQL = graph に対する問い合わせ


In [ ]:
# ?door みたいなものは 変数
# ?door は FireDoor
# その ?door が connectedTo している相手を ?zone
# つまり graph の中から
# ?door rdf:type FireDoor
# ?door connectedTo ?zone
# というパターンに合うものを探しています。
query = '''
PREFIX std: <http://example.org/standard/>

SELECT ?clause ?code ?modality ?prop ?min ?max ?targetClass
WHERE {
  ?clause a std:Clause ;
          std:clauseCode ?code ;
          std:hasModality ?modality ;
          std:requiresProperty ?prop .
  OPTIONAL { ?clause std:minCount ?min . }
  OPTIONAL { ?clause std:maxCount ?max . }
  OPTIONAL { ?clause std:requiresTargetClass ?targetClass . }
}
ORDER BY ?code
'''

sparql_rows = []
for row in g_standard.query(query):
    sparql_rows.append(tuple(str(x) if x is not None else None for x in row))

sparql_df = pd.DataFrame(sparql_rows, columns=['clause', 'code', 'modality', 'property', 'min', 'max', 'targetClass'])
display(sparql_df)


,clause,code,modality,property,min,max,targetClass
0,http://example.org/standard/Clause_5_2_1,5.2.1,http://example.org/standard/Shall,http://example.org/hasIdentifier,1,1,None
1,http://example.org/standard/Clause_5_2_2,5.2.2,http://example.org/standard/Shall,http://example.org/connectedTo,1,None,http://example.org/AlarmZone
2,http://example.org/standard/Clause_5_2_3,5.2.3,http://example.org/standard/Should,http://example.org/hasInspectionRecord,1,None,http://example.org/InspectionRecord


## 12. この notebook で学んだこと

この notebook は小規模ですが、

- **規格文そのものを graph 化**している
- ontology / clause / modality / required relation を分けている
- **条文グラフから validation specification を生成**している
- **artifact graph に対して clause-level verdict** を出している
- 結果を table に戻して、Python 側で集計できるようにしている

特に重要なのは、

- ontology / semantics 層
- SHACL validation 層
- verdict / reporting 層

を分けていることです。


## 13. 次にやるとよい拡張

1. **`shall / should / may` の3値化**
2. **`unknown` と `violated` を分ける verdict policy**
3. **cross-reference / exception clause** の導入
4. **text clause から graph への半自動抽出**
5. **results_graph から justification object を組み立てる**
6. **RDF graph を hetero graph に変換して GNN 前処理へつなぐ**

この方向に伸ばすと、knowledge representation / semantic validation / Python analytics がかなり一体で見えてきます。
